# 🗣️ Text to Speech — Version finale (cahier des charges)

Application Tkinter conforme au cahier des charges :

| Fonctionnalité | Détail |
|---|---|
| 🎙️ Choix de la voix | Voice 1, Voice 2, ... (voix système détectées) |
| ⏱️ Vitesse | Slow / Normal / Fast |
| ▶️ Speak | Lit le texte à voix haute |
| ⏹️ Stop | Arrête la lecture en cours |
| 🗑️ Clear | Efface tout le texte |
| 💾 Sauvegarde | Exporte en `.mp3` |
| 🌍 Langue | Français, Anglais, Arabe, etc. (mode gTTS) |
| 📂 Importer | Charge un fichier `.txt` dans la zone de texte |
| 🔢 Compteur | Nombre de mots et de caractères |
| 🕘 Historique | Derniers textes convertis |

**Dépendances à installer :**
```
pip install pyttsx3 gTTS pygame pydub
```
`pydub` nécessite [ffmpeg](https://ffmpeg.org/download.html) installé et accessible dans le PATH
(uniquement utile pour convertir la voix pyttsx3 en `.mp3` — sinon un `.wav` est proposé à la place).

⚠️ Exécutez les cellules **dans l'ordre, une seule fois**. Pour relancer : fermez la fenêtre puis
**Kernel > Restart & Run All**.

## 1. Imports et configuration du logger

In [ ]:
"""Text to Speech — application Tkinter de synthèse vocale (pyttsx3 + gTTS)."""

import os
import shutil
import logging
import tempfile
import threading
import datetime

import tkinter as tk
from tkinter import ttk, messagebox, filedialog, WORD

import pyttsx3
from gtts import gTTS
import pygame  # lecture audio pilotable (nécessaire pour le bouton Stop en mode gTTS)

try:
    from pydub import AudioSegment  # conversion .wav -> .mp3 (nécessite ffmpeg)
    HAS_PYDUB = True
except ImportError:
    HAS_PYDUB = False

try:
    import pythoncom  # Windows uniquement : requis pour pyttsx3 dans un thread
    HAS_PYTHONCOM = True
except ImportError:
    HAS_PYTHONCOM = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("text_to_speech")

pygame.mixer.init()
logger.info("Toutes les bibliothèques sont chargées avec succès.")

## 2. Constantes de configuration

In [ ]:
# Langues proposées pour le moteur gTTS : libellé affiché -> code ISO
GTTS_LANGUAGES: dict[str, str] = {
    "Français": "fr",
    "Anglais": "en",
    "Arabe": "ar",
    "Espagnol": "es",
    "Allemand": "de",
    "Italien": "it",
}

# Préréglages de vitesse pyttsx3 : libellé -> débit (mots/minute environ)
SPEED_PRESETS: dict[str, int] = {
    "Slow": 110,
    "Normal": 180,
    "Fast": 260,
}

# Palette "moderne"
COLOR_BG = "#F4F6F8"
COLOR_HEADER = "#2D6CDF"
COLOR_HEADER_TEXT = "#FFFFFF"
COLOR_ACCENT = "#2D6CDF"
COLOR_ACCENT_HOVER = "#1E52B5"
COLOR_DANGER = "#DC2626"
COLOR_DANGER_HOVER = "#B91C1C"
COLOR_SURFACE = "#FFFFFF"
COLOR_TEXT = "#1F2937"
COLOR_MUTED = "#6B7280"

FONT_TITLE = ("Segoe UI", 24, "bold")
FONT_LABEL = ("Segoe UI", 11, "bold")
FONT_TEXT = ("Segoe UI", 13)
FONT_SMALL = ("Segoe UI", 9)

## 3. Classe `TextToSpeechApp`

In [ ]:
class TextToSpeechApp:
    """Application de synthèse vocale (Tkinter + pyttsx3 + gTTS + pygame).

    Architecture :
        - ``_build_*``  : construisent les zones de l'interface.
        - ``_on_*``     : callbacks déclenchés par l'utilisateur.
        - ``_speak_*``  : logique de synthèse vocale pure.
    """

    def __init__(self, root: tk.Tk) -> None:
        """Initialise l'état interne puis construit toute l'interface."""
        self.root = root
        self.history: list[str] = []
        self._current_pyttsx3_engine: pyttsx3.Engine | None = None  # pour permettre Stop
        self._is_speaking = False

        self._setup_style()
        self._setup_window()
        self._build_header()
        self._build_text_area()
        self._build_controls()
        self._build_history_panel()
        self._build_status_bar()
        self._on_engine_change()

    # ------------------------------------------------------------------
    # 3.1 Construction de l'interface
    # ------------------------------------------------------------------
    def _setup_style(self) -> None:
        """Configure un thème ttk moderne et plat pour tous les widgets."""
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TCombobox", padding=4, font=FONT_TEXT)

        style.configure("Accent.TButton", font=("Segoe UI", 12, "bold"),
                        background=COLOR_ACCENT, foreground="white", padding=8, borderwidth=0)
        style.map("Accent.TButton", background=[("active", COLOR_ACCENT_HOVER)])

        style.configure("Danger.TButton", font=("Segoe UI", 12, "bold"),
                        background=COLOR_DANGER, foreground="white", padding=8, borderwidth=0)
        style.map("Danger.TButton", background=[("active", COLOR_DANGER_HOVER)])

        style.configure("Secondary.TButton", font=("Segoe UI", 10), padding=6,
                        background="#E5E7EB", foreground=COLOR_TEXT, borderwidth=0)
        style.map("Secondary.TButton", background=[("active", "#D1D5DB")])

        style.configure("TProgressbar", troughcolor="#E5E7EB", background=COLOR_ACCENT)

    def _setup_window(self) -> None:
        """Configure la fenêtre principale."""
        self.root.title("Text to Speech — Pro")
        self.root.geometry("1140x660+130+40")
        self.root.resizable(False, False)
        self.root.configure(bg=COLOR_BG)

    def _build_header(self) -> None:
        """Construit le bandeau supérieur."""
        header = tk.Frame(self.root, width=1140, height=80, bg=COLOR_HEADER)
        header.place(x=0, y=0)
        tk.Label(header, text="🗣️  Text to Speech", font=FONT_TITLE,
                 bg=COLOR_HEADER, fg=COLOR_HEADER_TEXT).place(x=390, y=20)

    def _build_text_area(self) -> None:
        """Construit la zone de texte, le bouton d'import de fichier, et le compteur."""
        tk.Label(self.root, text="Votre texte", font=FONT_LABEL, bg=COLOR_BG, fg=COLOR_TEXT).place(x=30, y=100)

        ttk.Button(self.root, text="📂 Importer un .txt", style="Secondary.TButton",
                   command=self._on_import_file).place(x=560, y=97, width=170, height=28)

        self.text_box = tk.Text(self.root, font=FONT_TEXT, bg=COLOR_SURFACE, fg=COLOR_TEXT,
                                 relief="flat", wrap=WORD, bd=0, highlightthickness=1,
                                 highlightbackground="#D1D5DB", highlightcolor=COLOR_ACCENT)
        self.text_box.place(x=30, y=130, width=700, height=220)
        self.text_box.bind("<KeyRelease>", self._update_counts)

        self.counter_label = tk.Label(self.root, text="0 mot(s) — 0 caractère(s)",
                                       font=FONT_SMALL, bg=COLOR_BG, fg=COLOR_MUTED)
        self.counter_label.place(x=30, y=355)

    def _build_controls(self) -> None:
        """Construit les contrôles : voix, vitesse, langue, et les boutons d'action."""
        y0 = 390

        # Voix (pyttsx3)
        tk.Label(self.root, text="Voix", font=FONT_LABEL, bg=COLOR_BG, fg=COLOR_TEXT).place(x=30, y=y0)
        self.voice_combo = ttk.Combobox(self.root, state="readonly")
        self.voice_combo.place(x=30, y=y0 + 25, width=210, height=30)
        self._refresh_voice_list()

        # Vitesse (Slow / Normal / Fast)
        tk.Label(self.root, text="Vitesse", font=FONT_LABEL, bg=COLOR_BG, fg=COLOR_TEXT).place(x=260, y=y0)
        self.speed_combo = ttk.Combobox(self.root, values=list(SPEED_PRESETS.keys()), state="readonly")
        self.speed_combo.current(1)  # "Normal" par défaut
        self.speed_combo.place(x=260, y=y0 + 25, width=140, height=30)

        # Langue (gTTS)
        tk.Label(self.root, text="Langue", font=FONT_LABEL, bg=COLOR_BG, fg=COLOR_TEXT).place(x=420, y=y0)
        self.lang_combo = ttk.Combobox(self.root, values=list(GTTS_LANGUAGES.keys()), state="readonly")
        self.lang_combo.current(0)
        self.lang_combo.place(x=420, y=y0 + 25, width=140, height=30)

        # Moteur (pyttsx3 utilise Voix + Vitesse, gTTS utilise Langue)
        tk.Label(self.root, text="Moteur", font=FONT_LABEL, bg=COLOR_BG, fg=COLOR_TEXT).place(x=580, y=y0)
        self.engine_combo = ttk.Combobox(
            self.root, values=["pyttsx3 (hors ligne)", "gTTS (Google, en ligne)"], state="readonly",
        )
        self.engine_combo.current(0)
        self.engine_combo.place(x=580, y=y0 + 25, width=150, height=30)
        self.engine_combo.bind("<<ComboboxSelected>>", self._on_engine_change)

        # Boutons d'action
        ttk.Button(self.root, text="▶️  Speak", style="Accent.TButton",
                   command=self._on_speak_clicked).place(x=30, y=y0 + 75, width=150, height=42)

        ttk.Button(self.root, text="⏹️  Stop", style="Danger.TButton",
                   command=self._on_stop_clicked).place(x=190, y=y0 + 75, width=150, height=42)

        ttk.Button(self.root, text="🗑️  Clear", style="Secondary.TButton",
                   command=self._on_clear_clicked).place(x=350, y=y0 + 75, width=150, height=42)

        ttk.Button(self.root, text="💾  Enregistrer (.mp3)", style="Secondary.TButton",
                   command=self._on_save_clicked).place(x=510, y=y0 + 75, width=220, height=42)

    def _build_history_panel(self) -> None:
        """Construit le panneau listant les derniers textes convertis."""
        tk.Label(self.root, text="Historique", font=FONT_LABEL, bg=COLOR_BG, fg=COLOR_TEXT).place(x=770, y=100)

        self.history_listbox = tk.Listbox(self.root, font=FONT_SMALL, relief="flat", bd=0,
                                           highlightthickness=1, highlightbackground="#D1D5DB",
                                           bg=COLOR_SURFACE)
        self.history_listbox.place(x=770, y=130, width=330, height=430)
        self.history_listbox.bind("<Double-Button-1>", self._on_history_double_click)

        tk.Label(self.root, text="Double-clic : recharger le texte", font=FONT_SMALL,
                 bg=COLOR_BG, fg=COLOR_MUTED).place(x=770, y=565)

    def _build_status_bar(self) -> None:
        """Construit la barre de statut et la barre de progression."""
        self.status_var = tk.StringVar(value="Prêt.")
        tk.Label(self.root, textvariable=self.status_var, font=FONT_SMALL, bg="#E5E7EB",
                 fg=COLOR_TEXT, anchor="w").place(x=0, y=635, width=1140, height=25)
        self.progress = ttk.Progressbar(self.root, mode="indeterminate", length=200)

    # ------------------------------------------------------------------
    # 3.2 Fonctions utilitaires d'interface
    # ------------------------------------------------------------------
    def _refresh_voice_list(self) -> None:
        """Liste les voix pyttsx3 disponibles et les affiche sous forme générique
        'Voice 1', 'Voice 2', ... (avec le nom système entre parenthèses).
        Stocke les objets voix réels dans ``self._voices`` pour un usage interne.
        """
        tmp_engine = pyttsx3.init()
        self._voices = tmp_engine.getProperty("voices") or []
        tmp_engine.stop()
        del tmp_engine

        if self._voices:
            labels = [f"Voice {i + 1} ({v.name})" for i, v in enumerate(self._voices)]
        else:
            labels = ["Voice 1 (par défaut)"]

        self.voice_combo["values"] = labels
        self.voice_combo.current(0)

    def _on_engine_change(self, event: tk.Event | None = None) -> None:
        """Callback : active/désactive Voix+Vitesse ou Langue selon le moteur choisi
        (les deux restent visibles, mais seul le combo pertinent est utile).
        """
        is_offline = self.engine_combo.get() == "pyttsx3 (hors ligne)"
        self.voice_combo.configure(state="readonly" if is_offline else "disabled")
        self.speed_combo.configure(state="readonly" if is_offline else "disabled")
        self.lang_combo.configure(state="disabled" if is_offline else "readonly")

    def _on_import_file(self) -> None:
        """Callback du bouton 'Importer un .txt' : ouvre un fichier texte et
        remplace le contenu de la zone de texte par son contenu.
        """
        path = filedialog.askopenfilename(filetypes=[("Fichiers texte", "*.txt"), ("Tous les fichiers", "*.*")])
        if not path:
            return
        try:
            with open(path, "r", encoding="utf-8", errors="replace") as f:
                content = f.read()
            self.text_box.delete("1.0", tk.END)
            self.text_box.insert("1.0", content)
            self._update_counts()
            self._set_status(f"Fichier importé : {os.path.basename(path)}")
        except Exception as exc:
            logger.exception("Erreur lors de l'import du fichier")
            messagebox.showerror("Erreur", f"Impossible d'ouvrir ce fichier :\n{exc}")

    def _update_counts(self, event: tk.Event | None = None) -> None:
        """Callback : met à jour le nombre de mots et de caractères en temps réel."""
        text = self._get_text()
        word_count = len(text.split()) if text else 0
        char_count = len(text)
        self.counter_label.config(text=f"{word_count} mot(s) — {char_count} caractère(s)")

    def _get_text(self) -> str:
        """Retourne le texte actuellement saisi (sans espaces superflus)."""
        return self.text_box.get("1.0", tk.END).strip()

    def _set_status(self, message: str, busy: bool = False) -> None:
        """Met à jour la barre de statut, avec ou sans barre de progression active."""
        self.status_var.set(message)
        if busy:
            self.progress.place(x=920, y=637, width=200, height=20)
            self.progress.start(10)
        else:
            self.progress.stop()
            self.progress.place_forget()

    def _add_to_history(self, text: str) -> None:
        """Ajoute un texte lu à l'historique de session (en mémoire)."""
        timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        preview = text[:45] + ("…" if len(text) > 45 else "")
        self.history.append(text)
        self.history_listbox.insert(tk.END, f"[{timestamp}] {preview}")

    def _on_history_double_click(self, event: tk.Event | None = None) -> None:
        """Callback : recharge dans la zone de texte l'entrée d'historique choisie."""
        selection = self.history_listbox.curselection()
        if not selection:
            return
        text = self.history[selection[0]]
        self.text_box.delete("1.0", tk.END)
        self.text_box.insert("1.0", text)
        self._update_counts()

    def _on_clear_clicked(self) -> None:
        """Callback du bouton 'Clear' : vide entièrement la zone de texte."""
        self.text_box.delete("1.0", tk.END)
        self._update_counts()

    # ------------------------------------------------------------------
    # 3.3 Logique de synthèse vocale
    # ------------------------------------------------------------------
    def _speak_pyttsx3(self, text: str, voice_index: int, rate: int) -> None:
        """Lit `text` avec pyttsx3. Un moteur neuf est créé à chaque appel et
        conservé dans ``self._current_pyttsx3_engine`` pour permettre au bouton
        Stop de l'interrompre. COM est initialisé/libéré dans ce thread (requis
        sous Windows, sinon aucun son ni erreur ne se produit).
        """
        if HAS_PYTHONCOM:
            pythoncom.CoInitialize()
        try:
            local_engine = pyttsx3.init()
            self._current_pyttsx3_engine = local_engine
            if self._voices and 0 <= voice_index < len(self._voices):
                local_engine.setProperty("voice", self._voices[voice_index].id)
            local_engine.setProperty("rate", rate)
            local_engine.say(text)
            local_engine.runAndWait()
            local_engine.stop()
        finally:
            self._current_pyttsx3_engine = None
            if HAS_PYTHONCOM:
                pythoncom.CoUninitialize()

    def _speak_gtts(self, text: str, lang_code: str) -> None:
        """Lit `text` avec gTTS : génère un mp3 temporaire puis le joue avec pygame
        (pygame permet d'interrompre la lecture via le bouton Stop, contrairement
        à `playsound` qui est bloquant et non interruptible).
        """
        tmp_path = os.path.join(tempfile.gettempdir(), "tts_output.mp3")
        gTTS(text=text, lang=lang_code).save(tmp_path)
        pygame.mixer.music.load(tmp_path)
        pygame.mixer.music.play()
        while pygame.mixer.music.get_busy():
            pygame.time.wait(100)

    # ------------------------------------------------------------------
    # 3.4 Callbacks des boutons principaux
    # ------------------------------------------------------------------
    def _on_speak_clicked(self) -> None:
        """Callback 'Speak' : valide le texte, l'ajoute à l'historique, puis lance
        la synthèse vocale dans un thread pour ne pas geler l'interface.
        """
        text = self._get_text()
        if not text:
            messagebox.showwarning("Texte vide", "Veuillez saisir un texte à lire.")
            return

        engine_choice = self.engine_combo.get()
        self._add_to_history(text)
        self._is_speaking = True
        self._set_status("Lecture en cours…", busy=True)

        def run() -> None:
            try:
                if engine_choice == "pyttsx3 (hors ligne)":
                    rate = SPEED_PRESETS.get(self.speed_combo.get(), 180)
                    self._speak_pyttsx3(text, self.voice_combo.current(), rate)
                else:
                    lang_code = GTTS_LANGUAGES.get(self.lang_combo.get(), "fr")
                    self._speak_gtts(text, lang_code)
                self.root.after(0, lambda: self._set_status("Prêt."))
            except Exception as exc:
                logger.exception("Erreur pendant la lecture")
                self.root.after(0, lambda: self._set_status("Erreur."))
                self.root.after(0, lambda: messagebox.showerror("Erreur", f"Impossible de lire le texte :\n{exc}"))
            finally:
                self._is_speaking = False

        threading.Thread(target=run, daemon=True).start()

    def _on_stop_clicked(self) -> None:
        """Callback 'Stop' : interrompt la lecture en cours, quel que soit le moteur."""
        if pygame.mixer.music.get_busy():
            pygame.mixer.music.stop()
        if self._current_pyttsx3_engine is not None:
            try:
                self._current_pyttsx3_engine.stop()
            except Exception:
                logger.exception("Erreur en arrêtant pyttsx3")
        self._set_status("Lecture arrêtée.")

    def _on_save_clicked(self) -> None:
        """Callback 'Enregistrer (.mp3)' : génère l'audio puis propose de le sauvegarder.

        - En mode gTTS : export direct en .mp3.
        - En mode pyttsx3 : génère un .wav puis le convertit en .mp3 via pydub/ffmpeg.
          Si ffmpeg n'est pas disponible, propose d'enregistrer en .wav à la place.
        """
        text = self._get_text()
        if not text:
            messagebox.showwarning("Texte vide", "Veuillez saisir un texte à lire.")
            return

        engine_choice = self.engine_combo.get()
        self._set_status("Génération du fichier audio…", busy=True)

        def run() -> None:
            try:
                if engine_choice == "gTTS (Google, en ligne)":
                    lang_code = GTTS_LANGUAGES.get(self.lang_combo.get(), "fr")
                    tmp_mp3 = os.path.join(tempfile.gettempdir(), "tts_save.mp3")
                    gTTS(text=text, lang=lang_code).save(tmp_mp3)
                    final_path, filetypes, default_ext = tmp_mp3, [("Fichier audio MP3", "*.mp3")], ".mp3"
                else:
                    if HAS_PYTHONCOM:
                        pythoncom.CoInitialize()
                    local_engine = pyttsx3.init()
                    if self._voices:
                        idx = self.voice_combo.current()
                        if 0 <= idx < len(self._voices):
                            local_engine.setProperty("voice", self._voices[idx].id)
                    local_engine.setProperty("rate", SPEED_PRESETS.get(self.speed_combo.get(), 180))
                    tmp_wav = os.path.join(tempfile.gettempdir(), "tts_save.wav")
                    local_engine.save_to_file(text, tmp_wav)
                    local_engine.runAndWait()
                    if HAS_PYTHONCOM:
                        pythoncom.CoUninitialize()

                    if HAS_PYDUB:
                        tmp_mp3 = os.path.join(tempfile.gettempdir(), "tts_save.mp3")
                        AudioSegment.from_wav(tmp_wav).export(tmp_mp3, format="mp3")
                        final_path, filetypes, default_ext = tmp_mp3, [("Fichier audio MP3", "*.mp3")], ".mp3"
                    else:
                        final_path, filetypes, default_ext = tmp_wav, [("Fichier audio WAV", "*.wav")], ".wav"
                        self.root.after(0, lambda: messagebox.showinfo(
                            "ffmpeg introuvable",
                            "pydub/ffmpeg n'est pas installé : le fichier sera enregistré en .wav au lieu de .mp3.",
                        ))

                def ask_save_path() -> None:
                    dest = filedialog.asksaveasfilename(defaultextension=default_ext, filetypes=filetypes)
                    if dest:
                        shutil.copyfile(final_path, dest)
                        self._set_status(f"Fichier enregistré : {dest}")
                        logger.info("Fichier audio enregistré : %s", dest)
                    else:
                        self._set_status("Prêt.")

                self.root.after(0, ask_save_path)
            except Exception as exc:
                logger.exception("Erreur pendant l'enregistrement")
                self.root.after(0, lambda: self._set_status("Erreur."))
                self.root.after(0, lambda: messagebox.showerror("Erreur", f"Impossible d'enregistrer :\n{exc}"))

        threading.Thread(target=run, daemon=True).start()

## 4. Lancement de l'application

⚠️ Bloque le kernel tant que la fenêtre est ouverte (comportement normal de `mainloop`).

In [ ]:
def main() -> None:
    """Point d'entrée : crée la fenêtre racine et lance l'application."""
    root = tk.Tk()
    app = TextToSpeechApp(root)
    root.mainloop()


main()